# CK-M starter notebook: the Modeler checkpoint

*Energy Systems, Step by Step* · Modeler tier · checkpoint CK-M · about 240 minutes.

This notebook gives you the data of both briefs and the supplied model of part B. It gives no formulation, no hints and no answers: those are on the checkpoint page, and open once all eleven tasks there are answered.

| Part | What you do here | About |
|---|---|---|
| A | Model variant: formulate and solve the water brief. Empirical variant: analyse the year file. | 120 min |
| B | Run the supplied model, find its planted defect, fix it, solve again. | 60 min |
| C | Check one of your results against an independent reference. | 45 min |

Enter your results in the checkpoint page's tasks as you go. **How to run it:** Google Colab (*File → Upload notebook*) or locally with Python 3.9 or later, NumPy and SciPy 1.9 or later. The empirical variant also needs `town_year.json` in this notebook's folder.

In [ ]:
import json, os
import numpy as np
import scipy
from scipy.optimize import linprog

print("NumPy", np.__version__, "· SciPy", scipy.__version__)
major, minor = (int(v) for v in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this notebook needs SciPy 1.9 or later (pip install -U scipy)"

## Part A, model variant: the water brief

Prairie Town's water utility pumps groundwater into a water tower, and the town draws its water from the tower. Find tomorrow's pumping schedule that meets the town's water use at the least electricity cost for the day.

*All numbers in this brief are illustrative.*

In [ ]:
# The water brief (illustrative). Hour t runs from t:00 to t+1:00.
water = [180, 160, 150, 150, 170, 250, 420, 610, 640, 560, 480, 450, 440, 430, 420, 430, 470, 560, 640, 650, 560, 430, 320, 230]        # m3 used in each hour
price = [0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.2, 0.2, 0.2, 0.2, 0.1, 0.06, 0.06]   # $ per kWh
KWH_PER_M3 = 0.5        # kWh of electricity per m3 pumped
PUMP_MAX = 900          # m3 per hour
TOWER_MIN, TOWER_MAX = 800, 4000   # m3
TOWER_START = 2000       # m3 at the start of the day; at least as much at the end
DEMAND_CHARGE = 0.40   # $ per kW of the day's highest hourly draw
assert len(water) == len(price) == 24
print(f"{sum(water):,} m3 used in the day; highest hour {max(water)} m3")

In [ ]:
# Your formulation (sets, inputs with units, decisions, objective, rules) as a comment, then your model.

## Part A, empirical variant: Prairie Town's peaks

How far can Prairie Town count on its wind and solar farms when it needs power most? Report what it can and cannot count on, and how far that depends on the dataset's scaling.

Download `town_year.json` from the checkpoint page and put it next to this notebook (in Colab: the Files panel, then Upload).

In [ ]:
path = "town_year.json"
if os.path.exists(path):
    year = json.load(open(path))
    print(year["meta"]["title"], "·", year["meta"]["source"])
    print("start_utc:", year["start_utc"], "· hours:", len(year["demand"]))
    print("Read year['meta'] before you scale anything: units, time convention, how bad hours were filled.")
else:
    print("town_year.json is not next to this notebook yet: download it from the checkpoint page.")

In [ ]:
# Your analysis design (question type, metrics with units, scaling, hours and ties, what it leaves out), then your code.

## Part B: the supplied model, with one planted defect

What should Prairie Town build (wind, solar, a gas combined-cycle plant, a gas peaker) to meet demand on three representative days of 2025, each weighted by the days of the year it stands for? On paper:

| Rule | Form |
|---|---|
| Weights | $w_d$ = the days of the year that day $d$ stands for (121, 122, 122; they sum to 365) |
| Goal | $\min \sum_g A_g K_g + \sum_d w_d \sum_t \big[\big(\sum_g c_g p_{g,d,t}\big) + V u_{d,t}\big]$ (\$ a year) |
| Balance | $\sum_g p_{g,d,t} + u_{d,t} = D_{d,t}$ (MW) |
| Wind and solar | $p_{g,d,t} \le a_{g,d,t} K_g$ |
| Gas plants | $p_{g,d,t} \le K_g$ |
| Domains | $K_g \ge 0$, $p_{g,d,t} \ge 0$, $u_{d,t} \ge 0$ |

| Symbol | Meaning | Unit |
|---|---|---|
| $g$ | technology: wind, solar, gas combined-cycle (cc), gas peaker (pk) | – |
| $d$, $t$ | representative day, and hour of it (one hour each) | – |
| $w_d$ | days of the year that day $d$ stands for | days |
| $A_g$ | capital cost, already annualized | \$ per MW of capacity per year |
| $K_g$ | capacity built (a decision) | MW |
| $c_g$ | running cost | \$/MWh |
| $p_{g,d,t}$, $u_{d,t}$ | output, and demand left unserved (decisions) | MW, held for one hour |
| $V$ | cost of unserved demand, \$10,000 | \$/MWh |
| $D_{d,t}$ | demand | MW |
| $a_{g,d,t}$ | output per MW built, 0 to 1 | MW per MW |

The code below runs as supplied. It solves to optimal. Is it right?

In [ ]:
# The supplied model's three representative days of 2025 (real SPP shapes; see the checkpoint page)
demand = {
    "winter": [277.65, 275.85, 275.85, 277.2, 281.7, 292.5, 310.05, 319.95, 315.45, 302.85, 292.05, 283.95, 275.85, 269.1, 264.15, 261, 263.25, 274.05, 284.85, 287.55, 288, 284.85, 277.2, 270],
    "shoulder": [231.3, 226.8, 223.65, 222.75, 224.55, 231.3, 244.8, 255.15, 257.85, 259.65, 261.45, 262.8, 261.9, 262.8, 262.8, 262.35, 263.7, 264.6, 263.25, 261, 263.25, 256.5, 244.8, 232.65],
    "summer": [257.85, 248.85, 242.55, 238.95, 239.85, 246.6, 261, 269.55, 278.1, 290.7, 308.7, 328.95, 348.75, 365.85, 377.1, 383.4, 386.55, 383.4, 373.5, 359.55, 347.85, 329.85, 308.7, 288.45],
}   # MW
avail = {
    "wind": {
        "winter": [0.4741, 0.4722, 0.4645, 0.4677, 0.4645, 0.4459, 0.4248, 0.3646, 0.289, 0.1768, 0.0942, 0.075, 0.0871, 0.1205, 0.1768, 0.2345, 0.3024, 0.3851, 0.469, 0.5049, 0.5279, 0.5324, 0.5331, 0.5209],
        "shoulder": [0.4421, 0.4235, 0.4145, 0.4062, 0.3921, 0.3652, 0.3466, 0.3332, 0.3678, 0.4158, 0.4414, 0.4286, 0.3998, 0.3806, 0.3614, 0.3441, 0.3453, 0.3139, 0.2774, 0.2102, 0.2031, 0.2236, 0.2383, 0.2665],
        "summer": [0.3562, 0.3364, 0.3312, 0.3389, 0.3319, 0.3261, 0.3184, 0.2966, 0.2351, 0.1403, 0.1211, 0.1307, 0.1378, 0.1454, 0.1538, 0.1647, 0.1877, 0.2178, 0.2242, 0.2518, 0.3453, 0.4338, 0.4812, 0.4754],
    },
    "solar": {
        "winter": [0, 0, 0, 0, 0, 0, 0, 0.001, 0.1158, 0.3996, 0.4939, 0.4447, 0.3617, 0.3709, 0.3494, 0.4324, 0.3084, 0.0717, 0, 0, 0, 0, 0, 0],
        "shoulder": [0, 0, 0, 0, 0, 0, 0, 0.0102, 0.2203, 0.4488, 0.4509, 0.5472, 0.5226, 0.5769, 0.6292, 0.5697, 0.497, 0.4355, 0.333, 0.0615, 0, 0, 0, 0],
        "summer": [0, 0, 0, 0, 0, 0, 0, 0.0236, 0.2285, 0.6486, 0.8249, 0.8474, 0.8136, 0.8074, 0.788, 0.8331, 0.789, 0.787, 0.4478, 0.0728, 0, 0, 0, 0],
    },
}   # output per MW built, 0 to 1

In [ ]:
DAYS = {"winter": 121, "shoulder": 122, "summer": 122}       # days of 2025 each day stands for
weight = {d: DAYS[d] / 365 for d in DAYS}                    # share of the year
cost = {"wind": 0, "solar": 0, "cc": 40, "pk": 100}           # running cost, $ per MWh
capital = {"wind": 100_000, "solar": 70_000, "cc": 80_000, "pk": 50_000}   # $ per MW-year, annualized
V = 10_000                                                    # $ per MWh of demand left unserved
G = ["wind", "solar", "cc", "pk"]

def supplied_model(weight):
    # variables: K[g] (MW built), then p[g,d,t] (MW) and u[d,t] (MW) for every day and hour
    names = [f"K_{g}" for g in G] + [f"{v}_{d}_{t}" for d in DAYS for t in range(24) for v in [f"p_{g}" for g in G] + ["u"]]
    col = {n: j for j, n in enumerate(names)}
    c = np.zeros(len(names))
    for g in G:
        c[col[f"K_{g}"]] = capital[g]
    for d in DAYS:
        for t in range(24):
            for g in G:
                c[col[f"p_{g}_{d}_{t}"]] = weight[d] * cost[g]
            c[col[f"u_{d}_{t}"]] = weight[d] * V
    A_eq, b_eq, A_ub = [], [], []
    for d in DAYS:
        for t in range(24):
            r = np.zeros(len(names))                                   # balance, MW
            for g in G:
                r[col[f"p_{g}_{d}_{t}"]] = 1
            r[col[f"u_{d}_{t}"]] = 1
            A_eq.append(r); b_eq.append(demand[d][t])
            for g in G:                                                # output within what is built
                r = np.zeros(len(names)); r[col[f"p_{g}_{d}_{t}"]] = 1
                r[col[f"K_{g}"]] = -(avail[g][d][t] if g in avail else 1)
                A_ub.append(r)
    res = linprog(c, A_ub=np.array(A_ub), b_ub=np.zeros(len(A_ub)), A_eq=np.array(A_eq), b_eq=b_eq, bounds=(0, None), method="highs")
    assert res.status == 0, res.message
    x = dict(zip(names, res.x))
    return dict(K={g: x[f"K_{g}"] for g in G}, annual_cost=res.fun,
                demand_in_year=sum(weight[d] * demand[d][t] for d in DAYS for t in range(24)),
                unserved_in_year=sum(weight[d] * x[f"u_{d}_{t}"] for d in DAYS for t in range(24)))

def report(R):
    for g, label in [("wind", "Wind farm"), ("solar", "Solar farm"), ("cc", "Gas combined-cycle"), ("pk", "Gas peaker")]:
        print(f"Built: {label:20s} {R['K'][g]:8.1f} MW")
    print(f"Annual cost: ${R['annual_cost'] / 1e6:.1f} million")
    print(f"Demand in the year: {R['demand_in_year']:,.0f} MWh")
    print(f"Demand left unserved in the year: {R['unserved_in_year']:,.0f} MWh")

supplied = supplied_model(weight)
report(supplied)

In [ ]:
# Your checks, the defect, your fix, and the fixed model's report.

## Part C: validation

Model variant: bound your optimum by hand (tasks 9 and 10) and reproduce one special case exactly. Empirical variant: compare one of your results with a published benchmark (tasks 9 and 10). Then write your validation statement in task 11 on the page.

In [ ]:
# Your validation.